# 02 · Comparing the race pace of two drivers

**Mission covered:** 5

**Question:** during the race, which driver was faster, by how much, and which one was more consistent?

**What we learn:** filtering a DataFrame, cleaning data, grouping with `groupby`,
computing statistics with numpy.

**Done when** you can write a sentence such as:
"On representative laps, X was 0.3 s per lap faster than Y on average, and more consistent."

**How to use this notebook:** each step starts with a ✏️ **Exercise**. Try to answer it yourself
(add a new cell below it) before reading the ✅ **Solution** and the 💡 **Explanation** that follow.

## Setup

These cells are the same as in `01_exploration.ipynb`: imports, FastF1 cache and the `show` function.
The exercises and explanations about them are in notebook 01.

> 🔁 Copying this block into every notebook is not ideal: if it is fixed in one place, it has to be fixed everywhere.
> This is exactly the problem that **mission 8** will solve (moving this code into `src/`).

In [ ]:
import os

import fastf1
import fastf1.plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
# The relative path "../cache" is only correct if the notebook runs from notebooks/
assert os.path.basename(os.getcwd()) == "notebooks", f"Unexpected working directory: {os.getcwd()}"

cache_dir = os.path.abspath("../cache")
os.makedirs(cache_dir, exist_ok=True)
print("FastF1 cache:", cache_dir)

fastf1.Cache.configure(cache_dir=cache_dir)

In [ ]:
def show(df, columns=None, n=None):
    '''Display a DataFrame as a Jupyter table, without truncated columns.

    columns : list of column names to keep (all by default)
    n       : number of rows to display (all by default)
    '''
    if columns is not None:
        df = df[columns]
    if n is not None:
        df = df.head(n)
    with pd.option_context("display.max_columns", None):
        display(df)

### Loading the session


We use the **race** (`"R"`) rather than qualifying: dozens of laps per driver are needed
to compare pace.

In [ ]:
YEAR = 2024
GRAND_PRIX = "Monaco"
SESSION_TYPE = "R"

session = fastf1.get_session(YEAR, GRAND_PRIX, SESSION_TYPE)
session.load()
print(session.event["EventName"], "-", session.name)

In [ ]:
# Without a Formula 1 account sign-in, FastF1 only retrieves the results:
# accessing session.laps then raises a DataNotLoadedError.
try:
    laps_count = len(session.laps)
except fastf1.exceptions.DataNotLoadedError:
    laps_count = 0

if laps_count == 0:
    print("⚠️ No laps loaded. Sign in from a terminal with:")
    print("   python -m fastf1 auth f1tv --authenticate")
else:
    print(f"✅ {laps_count} laps loaded")

## 5.1 · Choosing two drivers

> ✏️ **Exercise**
>
> Choose two drivers and store their three-letter abbreviations (`Driver` column) in constants.
>
> Which matchup is the most interesting? Two teammates (same car, so you really compare the drivers)
> or the top two of the race?

In [ ]:
# ✅ Solution
DRIVER_1 = "LEC"
DRIVER_2 = "PIA"
DRIVERS = [DRIVER_1, DRIVER_2]

💡 **Explanation**

Charles Leclerc (Ferrari) and Oscar Piastri (McLaren): the top two in both qualifying and the race,
which gives a direct battle for the win. With constants, changing the matchup only takes one line.

## 5.2 · Filtering their laps

> ✏️ **Exercise**
>
> Keep only the laps of the two drivers.
>
> 1. With **pandas**: boolean filtering keeps the rows where a condition is true.
>    ```python
>    students[students["class"] == "3B"]   # unrelated example: students in class 3B
>    ```
>    How do you keep **two** drivers at once? Look for `.isin()`.
> 2. With **FastF1**: find a method of `Laps` that does the same thing.
> 3. Check that both methods give the same number of laps.

In [ ]:
# ✅ Solution
laps_pandas = session.laps[session.laps["Driver"].isin(DRIVERS)]
laps_fastf1 = session.laps.pick_drivers(DRIVERS)

print("Laps with pandas:", len(laps_pandas))
print("Laps with FastF1:", len(laps_fastf1))

💡 **Explanation**

`.isin(DRIVERS)` returns `True` for every row whose driver is in the list. `pick_drivers` does exactly the same:
both give 156 laps (78 laps each).

## 5.3 · Cleaning: keeping representative laps

> ✏️ **Exercise**
>
> This is the most important step. Which laps distort a pace comparison? Remove them.
>
> Think about what you found in notebook 01:
> - laps with a pit stop;
> - laps behind the safety car or under a flag;
> - the first lap (standing start);
> - laps flagged as inaccurate or deleted.
>
> Hints: the `PitInTime`, `PitOutTime`, `TrackStatus`, `IsAccurate`, `Deleted` and `LapNumber` columns,
> and the `pick_wo_box`, `pick_track_status`, `pick_accurate`, `pick_not_deleted` methods in the FastF1 documentation.
>
> Then add a `LapTime_s` column: the lap time **in seconds** (look for `.dt.total_seconds()`).

In [ ]:
# ✅ Solution
laps = session.laps.pick_drivers(DRIVERS)

clean_laps = (
    laps
    .pick_wo_box()            # remove pit out-laps and in-laps
    .pick_track_status("1")   # keep only laps fully under green flag
    .pick_accurate()          # keep lap times FastF1 considers reliable
    .pick_not_deleted()       # remove laps deleted by race control (track limits)
)
clean_laps = clean_laps[clean_laps["LapNumber"] > 1]   # lap 1 (standing start) is not representative

# Lap time in seconds: a float is easier to work with than a duration
clean_laps = clean_laps.assign(LapTime_s=clean_laps["LapTime"].dt.total_seconds())

💡 **Explanation**

| Method | What it removes |
|---|---|
| `pick_wo_box()` | pit out-laps and in-laps |
| `pick_track_status("1")` | any lap not fully under green flag (safety car, yellow flag…) |
| `pick_accurate()` | laps whose time is not considered reliable |
| `pick_not_deleted()` | deleted laps |
| `LapNumber > 1` | the first lap, disrupted by the standing start |

Each method returns a new table, so they can be **chained**. The parentheses around the expression
allow one method per line.

> ✏️ **Exercise**
>
> How many laps were removed for each driver? Display a small table with the number of laps **before** and **after** cleaning.
> Then display the removed laps: why was each one removed?
>
> Hint: `~` means "not"; `laps.index.isin(clean_laps.index)` tells which rows were kept.

In [ ]:
# ✅ Solution
before_after = pd.DataFrame({
    "before": laps["Driver"].value_counts(),
    "after": clean_laps["Driver"].value_counts(),
})
show(before_after)

In [ ]:
# ✅ Solution
removed = laps[~laps.index.isin(clean_laps.index)]
show(removed, ["Driver", "LapNumber", "LapTime", "Compound", "TrackStatus", "PitInTime", "PitOutTime", "IsAccurate"])

💡 **Explanation**

Three laps were removed for each driver:
- **Lap 1**: the race was stopped by a **red flag** on the very first lap (`TrackStatus` contains `5`).
  Its "time" is over 40 minutes, because it includes the whole interruption.
- **Lap 2**: the restart, behind the **safety car** (`4`), with no usable lap time.
- **Lap 53**: a **yellow flag** (`TrackStatus` = `12`) during the lap. The time is not abnormal here,
  but drivers must slow down in the affected area: the lap is not comparable to the others.

Without this cleaning, a single 40-minute lap would completely distort the average.

## 5.4 · Statistics per driver

> ✏️ **Exercise**
>
> For each driver, compute the number of laps, the mean, the **median**, the standard deviation and the fastest lap.
>
> Hint: `groupby("Driver")` splits the table into one group per driver, and `.agg([...])` computes
> several statistics on each group in a single operation.

In [ ]:
# ✅ Solution
stats = clean_laps.groupby("Driver")["LapTime_s"].agg(["count", "mean", "median", "std", "min"]).round(3)
show(stats)

💡 **Explanation**

- **`mean`**: the average;
- **`median`**: the middle value, less sensitive to abnormally slow or fast laps;
- **`std`**: the standard deviation, which measures **consistency** (the smaller it is, the more consistent the driver);
- **`min`**: the fastest lap.

## 5.5 · Gaps with numpy

> ✏️ **Exercise**
>
> With numpy, compute the **median** and **mean** gap between the two drivers, and the standard deviation of each.
>
> 1. How do you turn a pandas column into a numpy array? Look for `.to_numpy()`.
> 2. What does a **negative** gap mean?
> 3. Why might the median be more reliable than the mean here?

In [ ]:
# ✅ Solution
times_1 = clean_laps.loc[clean_laps["Driver"] == DRIVER_1, "LapTime_s"].to_numpy()
times_2 = clean_laps.loc[clean_laps["Driver"] == DRIVER_2, "LapTime_s"].to_numpy()

median_gap = np.median(times_1) - np.median(times_2)
mean_gap = np.mean(times_1) - np.mean(times_2)

print(f"Median gap ({DRIVER_1} - {DRIVER_2}): {median_gap:+.3f} s per lap")
print(f"Mean gap   ({DRIVER_1} - {DRIVER_2}): {mean_gap:+.3f} s per lap")
print(f"Standard deviation {DRIVER_1}: {np.std(times_1):.3f} s   |   {DRIVER_2}: {np.std(times_2):.3f} s")

💡 **Explanation**

1. `.to_numpy()` turns a pandas column into a numpy array, on which `np.median`, `np.mean` and `np.std` are applied.
2. A **negative** gap means driver 1 is faster (their time is smaller).
3. A few unusual laps (very fast or very slow) pull the mean but barely move the median.

## 5.6 · Lap-by-lap comparison

> ✏️ **Exercise**
>
> Comparing overall averages mixes laps driven in different conditions (fuel load, tyre wear).
> A fairer comparison: put both drivers **side by side, lap by lap**, and keep only the laps where both have a representative time.
>
> 1. Build a table with one row per lap number and one column per driver. Look for `pivot_table`.
> 2. Compute the median lap-by-lap gap.
> 3. On how many laps is each driver faster?

In [ ]:
# ✅ Solution
per_lap = clean_laps.pivot_table(index="LapNumber", columns="Driver", values="LapTime_s").dropna()
lap_gap = per_lap[DRIVER_1] - per_lap[DRIVER_2]

print(f"Comparable laps: {len(per_lap)}")
print(f"Median lap-by-lap gap ({DRIVER_1} - {DRIVER_2}): {lap_gap.median():+.3f} s")
print(f"{DRIVER_1} faster on {(lap_gap < 0).sum()} laps, {DRIVER_2} on {(lap_gap > 0).sum()} laps")

💡 **Explanation**

`.dropna()` removes the laps where one of the two drivers has no representative time.
Comparing the same lap removes most of the effect of fuel load and tyre wear.

## Conclusion

> ✏️ **Exercise**
>
> Write your conclusion: who was faster, by how much, and who was more consistent?
> Is the gap consistent with the gap at the finish line? Does the standard deviation really measure consistency here?

💡 **Conclusion**

- **Leclerc won the race, Piastri finished 2nd, 7.2 s behind.** Over the 75 representative laps, their pace is
  almost identical: the median gap is only **0.017 s per lap** in Leclerc's favour.
- The **lap-by-lap** comparison, which is fairer, gives **0.075 s per lap** in Leclerc's favour;
  he is faster on 46 laps out of 75.
- The mean gap of **0.088 s per lap**, multiplied by 75 laps, gives about 6.6 s: consistent with
  the 7.2 s gap at the finish.
- **The mean and the median do not tell the same story**: the mean is pulled down by a few
  very fast laps from Leclerc at the end of the race (down to 1:15.2 on lap 71), while the median reflects the usual pace.
- **Limitation of the standard deviation**: here, it does not only measure consistency. Lap times drop from 84 s to 76 s
  over the race (as fuel burns off), and this trend inflates the standard deviation of both drivers.
  The two values (1.7 s and 1.6 s) are too close to name one driver as more consistent.